# Build DDPM From Scratch: Face Generation

You know the concepts. Now build it.

Implement DDPM (Ho et al. 2020) from scratch and train it to generate faces from pure noise.
Each section gives you the function signature and hints — you write the core logic.
Run the verification cells to check your work before moving on.

**Dataset:** CelebA 64x64 (30k subset for fast iteration)  
**Hardware:** RTX 2080 (8GB VRAM). Training: ~30-45 min for visible results.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

## Part 1: Data Loading

Before we touch diffusion, you need to get images into a training-ready format.

The pipeline: raw images on disk → transform to tensors at consistent size → scale pixel values to [-1, 1] → wrap in a DataLoader for batching.

**Why [-1, 1]?** The diffusion process adds Gaussian noise (mean 0). If your data is centered at 0, the noise blends naturally. If your data is [0, 1], the noisy images drift negative and the network sees a different distribution than expected.

**Your turn — three things to build:**

1. **Transform pipeline** — CelebA images are 178x218. You need to:
   - Center crop to square (178x178) so faces stay centered
   - Resize to 64x64 (small enough to train fast)
   - Convert to tensor
   - Normalize to [-1, 1]

2. **Dataset** — Load CelebA via `torchvision.datasets.CelebA`. Use a 30k subset for fast iteration.

3. **DataLoader** — Wrap dataset for batched training.

**Look up:** `transforms.CenterCrop`, `transforms.Resize`, `transforms.ToTensor`, `transforms.Normalize`, `transforms.Compose`, `torchvision.datasets.CelebA`, `torch.utils.data.Subset`, `DataLoader`

**Key detail for Normalize:** `Normalize([0.5]*3, [0.5]*3)` maps [0,1] → [-1,1] because it does `(x - mean) / std` = `(x - 0.5) / 0.5`.

In [ ]:
# EXERCISE: Build the data pipeline

IMG_SIZE = 64
CHANNELS = 3
BATCH_SIZE = 64
SUBSET_SIZE = 30_000

# YOUR CODE:
# 1. Build transform pipeline using transforms.Compose([...])
#    - CenterCrop(178)  — crop to square, faces are centered in CelebA
#    - Resize(IMG_SIZE)  — shrink to 64x64
#    - ToTensor()        — PIL image → torch tensor, also rescales [0,255] → [0,1]
#    - Normalize([0.5]*3, [0.5]*3)  — shift [0,1] → [-1,1]
transform = ...

# 2. Load CelebA dataset
#    datasets.CelebA(root="./data", split="train", download=True, transform=transform)
#    If download fails (Google Drive quota), download manually from:
#    https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html → place in ./data/celeba/
celeba = ...

# 3. Create a 30k subset for faster iteration
#    Subset(celeba, range(min(SUBSET_SIZE, len(celeba))))
dataset = ...

# 4. Create DataLoader for batched training
#    DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
#    - shuffle=True: randomize order each epoch (why? so the model doesn't memorize sequence)
#    - pin_memory=True: speeds up CPU→GPU transfer
#    - num_workers=2: parallel data loading
dataloader = ...

print(f"Dataset: {len(dataset)} images, shape: {dataset[0][0].shape}")
print(f"Batches per epoch: {len(dataloader)}")

In [ ]:
# VERIFY: images should be 64x64, values in [-1, 1], faces centered
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    img = dataset[i][0].permute(1, 2, 0).numpy() * 0.5 + 0.5  # back to [0,1] for display
    axes[i].imshow(img)
    axes[i].axis("off")
plt.suptitle("Training Samples (CelebA 64x64)")
plt.tight_layout()
plt.show()

# Sanity checks
sample = dataset[0][0]
assert sample.shape == (3, 64, 64), f"Expected (3, 64, 64), got {sample.shape}"
assert sample.min() >= -1.1 and sample.max() <= 1.1, f"Values should be in [-1, 1], got [{sample.min():.2f}, {sample.max():.2f}]"
print(f"Value range: [{sample.min():.2f}, {sample.max():.2f}]")
print("Data pipeline looks correct!")

# Check one batch from the dataloader
batch, _ = next(iter(dataloader))
assert batch.shape == (BATCH_SIZE, 3, 64, 64), f"Batch shape: expected ({BATCH_SIZE}, 3, 64, 64), got {batch.shape}"
print(f"Batch shape: {batch.shape}")

## Part 2: The Noise Schedule

Everything derives from one line: `betas = linspace(1e-4, 0.02, 1000)`.

From that, compute:
- `alphas` — how much signal survives each step (`1 - betas`)
- `alpha_bar` — how much *total* signal remains after t steps (cumulative product of alphas)

**Look up:** `torch.linspace`, `torch.cumprod`

In [ ]:
# EXERCISE: Build the noise schedule
T = 1000

# YOUR CODE: create betas, alphas, alpha_bar (all on device)
betas = ...       # (1000,) linearly spaced from 1e-4 to 0.02
alphas = ...      # (1000,) = 1 - betas
alpha_bar = ...   # (1000,) cumulative product of alphas

In [ ]:
# VERIFY: beta should increase linearly, alpha_bar should decay from ~1 to ~0
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ts = np.arange(T)

ax1.plot(ts, betas.cpu().numpy())
ax1.set(xlabel="Timestep", ylabel="beta[t]", title="Beta (noise per step)")
ax1.grid(True, alpha=0.3)

ax2.plot(ts, alpha_bar.cpu().numpy())
ax2.set(xlabel="Timestep", ylabel="alpha_bar[t]", title="Alpha Bar (signal retention)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

assert betas.shape == (T,), f"betas shape: expected ({T},), got {betas.shape}"
assert 0.99 < alpha_bar[0] < 1.0, f"alpha_bar[0] should be ~1.0, got {alpha_bar[0]:.4f}"
assert alpha_bar[-1] < 0.01, f"alpha_bar[-1] should be ~0, got {alpha_bar[-1]:.4f}"
print("Schedule looks correct!")

## Part 3: The Forward Process

The "teleport to any noise level" formula:

`x_noisy = sqrt(alpha_bar[t]) * x_0 + sqrt(1 - alpha_bar[t]) * noise`

Takes a clean image + timestep, returns the noisy version in one operation.

**The tricky part:** `t` is a batch of integers `(B,)`, but images are `(B, C, H, W)`.
You need to reshape the coefficients for broadcasting.

**Hint:** if `alpha_bar[t]` gives shape `(B,)`, use `.view(-1, 1, 1, 1)` to get `(B, 1, 1, 1)`.

In [ ]:
# EXERCISE: Implement the forward process

def q_sample(x0: torch.Tensor, t: torch.Tensor, noise: torch.Tensor | None = None) -> torch.Tensor:
    """Add noise to clean images x0 at timesteps t.

    The reparameterization trick: x_t = sqrt(alpha_bar_t) * x0 + sqrt(1 - alpha_bar_t) * noise

    Args:
        x0: clean images (B, C, H, W)
        t: timesteps (B,), values in [0, T)
        noise: optional pre-sampled noise, same shape as x0

    Returns:
        Noisy images at timesteps t (B, C, H, W)
    """
    if noise is None:
        noise = torch.randn_like(x0)

    # YOUR CODE: apply the reparameterization trick using alpha_bar schedule
    raise NotImplementedError


In [ ]:
# VERIFY: watch a face dissolve into noise
sample_img = dataset[0][0].unsqueeze(0).to(device)
noise = torch.randn_like(sample_img)

timesteps_to_show = [0, 50, 100, 200, 400, 700, 999]
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(16, 3))
for i, t_val in enumerate(timesteps_to_show):
    t_tensor = torch.tensor([t_val], device=device)
    noisy = q_sample(sample_img, t_tensor, noise=noise)
    img = noisy.squeeze().permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5
    axes[i].imshow(np.clip(img, 0, 1))
    axes[i].set_title(f"t={t_val}")
    axes[i].axis("off")

fig.suptitle("Forward Process: Face → Noise")
plt.tight_layout()
plt.show()

## Part 4: Timestep Embedding

The integer `t` needs to become a rich vector so the network knows its behavior should differ
at t=50 vs t=950. We use sinusoidal embedding (same idea as transformer positional encoding):
create waves at different frequencies, so nearby timesteps get similar vectors.

**Formula for each frequency dimension `i`:**

    freq[i] = exp(-i * log(10000) / (half_dim - 1))
    first half:  sin(t * freq[i])
    second half: cos(t * freq[i])

**Look up:** `torch.exp`, `torch.arange`, `torch.cat`

In [ ]:
# EXERCISE: Implement sinusoidal timestep embedding

class SinusoidalPosEmb(nn.Module):
    """Convert integer timestep to a vector of sines and cosines.

    Same encoding as the original Transformer position encoding,
    but applied to scalar timesteps instead of sequence positions.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """Args: t shape (B,). Returns: embedding shape (B, dim)."""
        # YOUR CODE: compute sinusoidal embedding
        # Output should be (B, dim) with sin in first half, cos in second half
        raise NotImplementedError


In [ ]:
# VERIFY: should see smooth wave patterns — nearby timesteps look similar
emb_module = SinusoidalPosEmb(dim=128)
all_t = torch.arange(T)
emb_matrix = emb_module(all_t).detach().numpy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(emb_matrix.T, aspect="auto", cmap="RdBu", origin="lower")
ax.set(xlabel="Timestep t", ylabel="Embedding dim", title="Sinusoidal Timestep Embeddings")
plt.colorbar(ax.images[0], ax=ax)
plt.tight_layout()
plt.show()

test_emb = emb_module(torch.tensor([0, 1, 500, 999]))
assert test_emb.shape == (4, 128), f"Expected (4, 128), got {test_emb.shape}"
print("Embedding looks correct!")

## Part 5: The U-Net

The noise predictor. Takes `(noisy_image, t)` → predicts the noise (same shape as image).

Architecture: encoder shrinks spatial dims while growing channels, decoder mirrors it back.
**Skip connections** concatenate encoder features to decoder features at matching resolutions.

```
Input (3, 64, 64)
  ↓ conv
Encoder: (64, 64, 64) → (128, 32, 32) → (256, 16, 16) → (256, 8, 8)
  ↓
Mid:     (256, 8, 8)  [ResBlock + Attention + ResBlock]
  ↓
Decoder: (256, 8, 8) → (256, 16, 16) → (128, 32, 32) → (64, 64, 64)
  ↓ conv                  ↑ skip            ↑ skip          ↑ skip
Output (3, 64, 64)
```

**Three exercises below:**
1. `ResBlock` — the core building block with timestep injection (you implement this)
2. `SelfAttention`, `Downsample`, `Upsample` — given (standard components)
3. `UNet.forward()` — you wire the layers together with skip connections

In [ ]:
# EXERCISE: Implement the residual block with time embedding injection
#
# Data flow:
#   x → conv1 → norm1 → silu → ADD time_proj(t_emb) → conv2 → norm2 → silu → ADD residual

class ResBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, time_emb_dim: int):
        super().__init__()
        # YOUR CODE: define conv1, norm1, conv2, norm2, time_proj, residual_conv
        raise NotImplementedError

    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        """x: (B, in_ch, H, W), t_emb: (B, time_emb_dim). Returns: (B, out_ch, H, W)."""
        # YOUR CODE: wire the layers together with time injection and residual
        raise NotImplementedError


In [ ]:
# GIVEN: Standard building blocks (not DDPM-specific)

class SelfAttention(nn.Module):
    """QKV self-attention over spatial dimensions."""

    def __init__(self, channels: int):
        super().__init__()
        self.norm = nn.GroupNorm(8, channels)
        self.qkv = nn.Conv2d(channels, channels * 3, 1)
        self.proj_out = nn.Conv2d(channels, channels, 1)
        self.scale = channels ** -0.5

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        h = self.norm(x)
        qkv = self.qkv(h).reshape(B, 3, C, H * W)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        attn = torch.bmm(q.permute(0, 2, 1), k) * self.scale
        attn = attn.softmax(dim=-1)
        out = torch.bmm(v, attn.permute(0, 2, 1)).reshape(B, C, H, W)
        return x + self.proj_out(out)


class Downsample(nn.Module):
    """Halve spatial resolution with strided conv."""
    def __init__(self, ch: int):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(x)


class Upsample(nn.Module):
    """Double spatial resolution with nearest interpolation + conv."""
    def __init__(self, ch: int):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv(F.interpolate(x, scale_factor=2, mode="nearest"))

In [ ]:
# EXERCISE: Wire up the U-Net
#
# __init__ is given — all layers are defined for you.
# Your job: implement forward() to connect them with skip connections.
#
# The key insight: during encoding, SAVE the output at each resolution.
# During decoding, CONCATENATE the saved features with the upsampled features
# using torch.cat([upsampled, saved], dim=1)  (along channel dim).

class UNet(nn.Module):
    def __init__(self, img_channels: int = 3, base_ch: int = 64, time_emb_dim: int = 256):
        super().__init__()
        # Time embedding: int → 256-dim vector via MLP
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_emb_dim),
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.SiLU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        ch = base_ch  # 64
        self.init_conv = nn.Conv2d(img_channels, ch, 3, padding=1)  # (3,64,64) → (64,64,64)

        # Encoder
        self.down1_res = ResBlock(ch, ch, time_emb_dim)          # (64,64,64) → (64,64,64)
        self.down1_attn = SelfAttention(ch)
        self.down1_pool = Downsample(ch)                          # → (64,32,32)

        self.down2_res = ResBlock(ch, ch * 2, time_emb_dim)      # (64,32,32) → (128,32,32)
        self.down2_pool = Downsample(ch * 2)                      # → (128,16,16)

        self.down3_res = ResBlock(ch * 2, ch * 4, time_emb_dim)  # (128,16,16) → (256,16,16)
        self.down3_attn = SelfAttention(ch * 4)
        self.down3_pool = Downsample(ch * 4)                      # → (256,8,8)

        # Bottleneck
        self.mid_res1 = ResBlock(ch * 4, ch * 4, time_emb_dim)   # (256,8,8)
        self.mid_attn = SelfAttention(ch * 4)
        self.mid_res2 = ResBlock(ch * 4, ch * 4, time_emb_dim)

        # Decoder (input channels doubled from skip connection concatenation)
        self.up3_res = ResBlock(ch * 4 * 2, ch * 4, time_emb_dim)  # (512,8,8) → (256,8,8)
        self.up3_attn = SelfAttention(ch * 4)
        self.up3_up = Upsample(ch * 4)                              # → (256,16,16)

        self.up2_res = ResBlock(ch * 4 + ch * 2, ch * 2, time_emb_dim)  # (256+128) → (128)
        self.up2_up = Upsample(ch * 2)                                    # → (128,32,32)

        self.up1_res = ResBlock(ch * 2 + ch, ch, time_emb_dim)   # (128+64) → (64)
        self.up1_attn = SelfAttention(ch)

        self.final_norm = nn.GroupNorm(8, ch)
        self.final_conv = nn.Conv2d(ch, img_channels, 1)  # (64,64,64) → (3,64,64)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """x: (B, 3, 64, 64), t: (B,). Returns: (B, 3, 64, 64)."""
        # YOUR CODE: encode with skip connections, bottleneck, decode with concatenation
        raise NotImplementedError


In [ ]:
# VERIFY: model should take (B, 3, 64, 64) + (B,) → (B, 3, 64, 64)
model = UNet(img_channels=CHANNELS).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,} ({total_params * 4 / 1024**2:.1f} MB)")

with torch.no_grad():
    dummy_x = torch.randn(2, CHANNELS, IMG_SIZE, IMG_SIZE, device=device)
    dummy_t = torch.randint(0, T, (2,), device=device)
    out = model(dummy_x, dummy_t)
    assert out.shape == dummy_x.shape, f"Shape mismatch: {out.shape} vs {dummy_x.shape}"
    print(f"Forward pass: {dummy_x.shape} → {out.shape}")
    print("U-Net works!")

# EXERCISE: Implement the training loop

EPOCHS = 15
LR = 2e-4
CKPT_DIR = Path("./checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

# Uses the dataloader you built in Part 1
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

losses = []
model.train()

for epoch in range(EPOCHS):
    epoch_losses = []
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for batch_imgs, _ in pbar:
        batch_imgs = batch_imgs.to(device)
        B = batch_imgs.shape[0]

        # YOUR CODE:
        # 1. Sample random timesteps t: randint(0, T, (B,)) on device
        # 2. Sample noise: randn_like(batch_imgs)
        # 3. Create noisy images: q_sample(batch_imgs, t, noise)
        # 4. Predict noise: model(noisy_images, t)
        # 5. Compute loss: F.mse_loss(predicted, actual noise)
        # 6. optimizer.zero_grad(), loss.backward(), optimizer.step()

        raise NotImplementedError

        epoch_losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = np.mean(epoch_losses)
    losses.append(avg_loss)
    print(f"Epoch {epoch + 1} — avg loss: {avg_loss:.4f}")

# Save checkpoint
torch.save({"model": model.state_dict(), "epoch": EPOCHS, "losses": losses},
           CKPT_DIR / "ddpm_celeba.pt")

plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# EXERCISE: Implement the training loop

BATCH_SIZE = 64
EPOCHS = 15
LR = 2e-4
CKPT_DIR = Path("./checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

losses = []
model.train()

for epoch in range(EPOCHS):
    epoch_losses = []
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{EPOCHS}")

    for batch_imgs, _ in pbar:
        batch_imgs = batch_imgs.to(device)

        # YOUR CODE: sample timesteps, add noise, predict noise, compute MSE loss, optimize
        raise NotImplementedError

        epoch_losses.append(loss.item())
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = np.mean(epoch_losses)
    losses.append(avg_loss)
    print(f"Epoch {epoch + 1} — avg loss: {avg_loss:.4f}")

torch.save({"model": model.state_dict(), "epoch": EPOCHS, "losses": losses},
           CKPT_DIR / "ddpm_celeba.pt")

plt.figure(figsize=(8, 4))
plt.plot(range(1, EPOCHS + 1), losses, marker="o")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.grid(True, alpha=0.3)
plt.show()


## Part 7: Sampling (Reverse Process)

Start from pure noise, count down from t=999 to t=0. At each step:

1. Model predicts noise: `eps_pred = model(x, t)`
2. Compute denoised estimate (the formula from the paper):
   `mu = (1/sqrt(alpha[t])) * (x - (beta[t] / sqrt(1 - alpha_bar[t])) * eps_pred)`
3. If t > 0, add fresh noise: `x = mu + sqrt(beta[t]) * random_noise`
4. If t = 0, just `x = mu` (final clean output)

**Look up:** `torch.no_grad` (we're doing inference, no gradients needed)

In [ ]:
# EXERCISE: Implement DDPM sampling

@torch.no_grad()
def ddpm_sample(
    model: nn.Module,
    n_samples: int = 16,
    save_every: int = 100,
) -> tuple[torch.Tensor, list[torch.Tensor]]:
    """Generate images by reversing the diffusion process.

    At each timestep t (from T-1 down to 0), predict the noise,
    compute the denoised mean, and add stochasticity (except at t=0).

    Returns:
        final_samples: (n_samples, C, H, W)
        intermediates: list of snapshots for visualization
    """
    model.eval()
    intermediates = []

    # Start from pure noise
    x = torch.randn(n_samples, CHANNELS, IMG_SIZE, IMG_SIZE, device=device)
    intermediates.append(x.clone())

    for t_val in tqdm(reversed(range(T)), total=T, desc="Sampling"):
        t_batch = torch.full((n_samples,), t_val, device=device, dtype=torch.long)

        # YOUR CODE: predict noise, compute posterior mean, add noise if t > 0
        raise NotImplementedError

        if t_val % save_every == 0:
            intermediates.append(x.clone())

    return x, intermediates


In [ ]:
# VERIFY: generate and display faces
samples, intermediates = ddpm_sample(model, n_samples=16)

grid = make_grid(samples.clamp(-1, 1), nrow=4, normalize=True, value_range=(-1, 1))
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.title("Generated Faces (DDPM)")
plt.axis("off")
plt.tight_layout()
plt.show()

# Denoising trajectory: watch a face emerge from noise
fig, axes = plt.subplots(1, len(intermediates), figsize=(2 * len(intermediates), 2))
for i, snap in enumerate(intermediates):
    img = snap[0].permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5
    axes[i].imshow(np.clip(img, 0, 1))
    step = 999 - i * 100 if i > 0 else 999
    axes[i].set_title(f"t={max(step, 0)}")
    axes[i].axis("off")
fig.suptitle("Reverse Process: Noise → Face")
plt.tight_layout()
plt.show()

## Bonus: Cosine Schedule

The linear beta schedule wastes timesteps — signal is destroyed too quickly in the middle.
The cosine schedule (Nichol & Dhariwal 2021) smooths this out so alpha_bar decays more
gradually, giving the model more useful training signal across all timesteps.

**If you want to go further:** implement the cosine schedule, retrain, and compare sample
quality side-by-side. The formula:

    alpha_bar[t] = cos((t/T + s) / (1 + s) * pi/2)^2    (s = 0.008)
    betas = 1 - alpha_bar[t] / alpha_bar[t-1]            (clamp to [0.0001, 0.999])

**Further reading:**
- Improved DDPM: [arXiv:2102.09672](https://arxiv.org/abs/2102.09672)
- DDIM (your "skip steps" idea): [arXiv:2010.02502](https://arxiv.org/abs/2010.02502)